<center> 

**Text Mining Project** <br>
Master in Data Science and Advanced Analytics <br>

NOVA Information Management School <br>
Universidade Nova de Lisboa 
</center>

<center> 
<b><font size="6" color="#0ee071">Tweets-Based Market Sentiment</font></b> 
</center>

<font color="#0ee071"> **Group 25 - June 2025** </font>
*   Beatris Daicu, 20221854
*   Diogo Carvalho, 20221935
*   Ricardo Pereira, 20250343
*   Yehor Malakhov, 20221691

<b><font size="5" color="#0ee071"> Introduction </font></b> 

Over time, major indexes go up and down based on internal and external factors. Performance like that excites investors, but typically in opposite ways. Constant gains lead some investors to
expect more of the same. Others worry the good times are surely about to end. The former sentiment is sometimes called **bullish**, while the latter is referred to as **bearish**.

The goal of this project is to develop an NLP model capable of predicting Market sentiment based on tweets, that is predicting the variable `label`, where it describes a **Bearish** (0), **Bullish** (1), or **Neutral** (2) attitude.

### <font color='#0ee071'>Table of Contents </font> <a class="anchor" id='index'></a> 

- [I. **Data Exploration**](#P1)
    - [Data Overview](#P1.1)
    - [Text Overview](#P1.2)
        - [Raw Text](#P1.2.1)
        - [Processed Text](#P1.2.2)
        - [Encoded Text](#P1.2.3)
        - [Encoded Text Vizualization](#P1.2.4)
- [II. **Classification Models**](#P2) 
    - [Bag of Words Data](#P2.1)
        - [KNN Model](#P2.1.1)
        - [Random Forest Model](#P2.1.2)
        - [Model Analysis](#P2.1.3)
    - [Word2Vec Data](#P2.2)
        - [KNN Model](#P2.2.1)
        - [Random Forest Model](#P2.2.2)
        - [Model Analysis](#P2.2.3)
    - [Transformers Data](#P2.3)
        - [KNN Model](#P2.3.1)
        - [Random Forest Model](#P2.3.2)
        - [Model Analysis](#P2.3.3)
    - [Transformer Classifier](#P2.4)


In [ ]:
%load_ext autoreload
%autoreload 2

import nltk

nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

import umap
import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils.encoders import *
from utils.preprocess import text_cleaner, pipeline
from utils.utils import visualize_dimensionality_reduction
from utils.transformer_classifier import train_model, predict

from lingua import LanguageDetectorBuilder
from deep_translator import GoogleTranslator
from wordcloud import WordCloud
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, recall_score
from sklearn.model_selection import ParameterSampler

import warnings
warnings.filterwarnings("ignore")

In [ ]:
STOPWORDS = set(stopwords.words("English")).difference({"up", "down", "above", "below", "under", 
                                                        "over", "more", "less", "against", "but", 
                                                        "not", "no"})

KNN_GRID = {
        "n_neighbors": [*range(1, 16, 2)],
        "weights": ["uniform", "distance"],
        "p": [1, 2, 3]
        }

RF_GRID = {
        "n_estimators":[*range(10, 251, 10)], 
        "criterion":["gini", "entropy"],
        "max_depth":[*range(1, 11)],
        "min_samples_split":[2**p for p in range(1, 11)],
        "min_samples_leaf":[2**p for p in range(0, 11)],
        "max_features":["sqrt", "log2", 1],
        }

SEED = 42

# <font color="#0ee071"> I. Data Exploration <a class="anchor" id="P1"></a></font>

[Back to Index](#index)

This initial phase focuses on understanding the data and vizualizing insights found in it. To simplify further work we decided to immediatly perform the **Corpus Split**. Out of original `train_full` data 80% went onto being used for training and 20% to be used as a validation data.

In [ ]:
train_full = pd.read_csv(r"data/train.csv")
train, validation = train_test_split(train_full, 
                                     test_size=0.2, 
                                     stratify=train_full.label,
                                     random_state=SEED
                                     )
test = pd.read_csv(r"data/test.csv", index_col=0)
display(train.sample(3))
display(validation.sample(3))
test.sample(3)

## <font color="#0ee071"> Data Overview <a class="anchor" id="P1.1"></a></font>

[Back to Index](#index)

In this chapter we only acess the data that is independent of the content of the corpora. Note absence of missing values and a large ammount of Neutral texts.

In [ ]:
display(train.describe())
train.describe(include="str")

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].bar(x = ["Bearish", "Bullish", "Neutral"],
        height=train.groupby("label")["text"].count().to_list(), # order by 0, 1, 2
        color="#00004A",
        )

ax[1].bar(x = ["Bearish", "Bullish", "Neutral"],
        height=validation.groupby("label")["text"].count().to_list(), # order by 0, 1, 2
        color="#4A0000",
        )

ax[0].set_xlabel("Label")
ax[0].set_ylabel("Count")
ax[0].set_title(
    "Train Label Distribution", fontweight="bold"
)

ax[1].set_xlabel("Label")
ax[1].set_ylabel("Count")
ax[1].set_title(
    "Validation Label Distribution", fontweight="bold"
)

plt.tight_layout()
plt.savefig(r"plots/correct_split.png")
plt.show()

## <font color="#0ee071"> Text Overview <a class="anchor" id="P1.2"></a></font>

[Back to Index](#index)

In this chapter we will overview data on different stages of our pipeline:
- before preprocessing (raw state)
- after preprocessing (clean state)
- after encoding


For first two states we overviewed text distributions via WordClouds an Bar Charts. The Encoded State was analysed using UMAP, to simplify the complexity of such high-dimentional data.

### <font color="#0ee071"> Raw Text <a class="anchor" id="P1.2.1"></a></font>

[Back to Index](#index)

As can be seen at first corpora has a large ammount of stopwords, which is to be expected and has a bimodal distribution of length of text, peaking at cerca 60 and 140 elements.

In [ ]:
freq_raw = pd.Series(' '.join(train['text']).split()).value_counts()

fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].hist(
         train.text.str.len(),
         bins=40,
         color="#00004A",
         edgecolor="white"
         )

ax[1].bar(freq_raw.head(10).index, freq_raw.head(10).values, color="#00004A")

ax[0].set_xlabel("Text Length")
ax[0].set_ylabel("Count")
ax[0].set_title("Text Length Distribution (raw)", fontweight="bold")
ax[1].set_xlabel("Word")
ax[1].set_ylabel("Count")
ax[1].set_title("Top 10 Most Frequent Words (raw)", fontweight="bold")
plt.tight_layout()
plt.savefig(r"plots/raw_distributions.png")
plt.show()

In [ ]:
raw_text = ' '.join(train['text']).lower()
raw_wc = WordCloud(width=800, 
                   height=400, 
                   background_color='white', 
                   collocations=False, 
                   random_state=SEED).generate(raw_text)

plt.figure(figsize=(10, 5))
plt.imshow(raw_wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud — Raw Text')
plt.savefig(r"plots/raw_wordcloud.png")
plt.show()

### <font color="#0ee071"> Processed Text <a class="anchor" id="P1.2.2"></a></font>

[Back to Index](#index)

In this part we cleaned the corpora in following manner:
1. Apply Cleaning Techniques from `utils.preprocessing.text_cleaner`
2. Detect Non-English Observations in Corpora
3. Translate Non-English Observations
4. Apply Cleaning Techniques from `utils.preprocessing.text_cleaner` to Translated Text. \
(N.B. Translator might change puntuation and formatting)
5. Tokenize Final Text and Perform Final Transformations (i.e. stopwords removal and lemmatization)

In [ ]:
train["text_clean"] = text_cleaner(train.text)
validation["text_clean"] = text_cleaner(validation.text)

display(train.sample(5))
validation.sample(5)

In [ ]:
detector = LanguageDetectorBuilder.from_all_languages().build()

languages_train = detector.detect_languages_in_parallel_of(train.text_clean)
train["language"] = [lang.name.lower() if lang else None for lang in languages_train]

languages_validation = detector.detect_languages_in_parallel_of(validation.text_clean)
validation["language"] = [lang.name.lower() if lang else None for lang in languages_validation]

non_eng_train = train[train["language"]!="english"]["text_clean"].index.to_list()
non_eng_validation = validation[validation["language"]!="english"]["text_clean"].index.to_list()

train.sample(5)
validation.sample(5)

In [ ]:
# GoogleTranslator sometimes adds symbols on its own, so has to be cleaned again even if use text_clean
# TODO use text_clean or text? 
# NB probably stochastic and has no seed

train.loc[non_eng_train, "text_clean"] = text_cleaner(
    pd.Series(
        GoogleTranslator().translate_batch(
            train.loc[non_eng_train, "text"].to_list()
                                        )
            )
    ).to_list()

validation.loc[non_eng_validation, "text_clean"] = text_cleaner(
    pd.Series(
        GoogleTranslator().translate_batch(
            validation.loc[non_eng_validation, "text"].to_list()
                                        )
            )
    ).to_list()

In [ ]:
train["text_tokens"] = pipeline(train["text_clean"], to_clean=False, stopwords_list=STOPWORDS)
validation["text_tokens"] = pipeline(validation["text_clean"], to_clean=False, stopwords_list=STOPWORDS)

As expected the length distribution changed with observation getting shorter. Notice still present two modes, one is still at cerca 60, but other lowered to 110. Concerning top words, the most common two words became the handlers we used to replace **urls** and **stock references**.

In [ ]:
train["text_clean_joined"] = train['text_tokens'].apply(lambda x: " ".join(x))
validation["text_clean_joined"] = validation['text_tokens'].apply(lambda x: " ".join(x))

freq_clean = pd.Series(' '.join(train['text_clean_joined']).split()).value_counts()

fig, ax = plt.subplots(ncols=2, figsize=(14, 5))
ax[0].hist(
         train.text_clean.str.len(),
         bins=40,
         color="#00004A",
         edgecolor="white"
         )
ax[1].bar(freq_clean.head(10).index, freq_clean.head(10).values, color="#00004A")

ax[0].set_xlabel("Text Length")
ax[0].set_ylabel("Count")
ax[0].set_title("Text Length Distribution (clean)", fontweight="bold")
ax[1].set_xlabel("Word")
ax[1].set_ylabel("Count")
ax[1].set_title("Top 10 Most Frequent Words (clean)", fontweight="bold")
plt.tight_layout()
plt.savefig(r"plots/clean_distributions.png")
plt.show()

In [ ]:
raw_text = ' '.join(train["text_clean_joined"]).lower()
raw_wc = WordCloud(width=800, 
                   height=400, 
                   background_color='white', 
                   collocations=False, 
                   random_state=SEED).generate(raw_text)

plt.figure(figsize=(10, 5))
plt.imshow(raw_wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud — Clean Text')
plt.savefig(r"plots/clean_wordcloud.png")
plt.show()

### <font color="#0ee071"> Encoded Text <a class="anchor" id="P1.2.3"></a></font>

[Back to Index](#index)

In [ ]:
train_bow, bow_encoder = encode_bow(train["text_tokens"], min_df=30)

w2v_encoder = train_word2vec_model(train["text_tokens"], vector_size=35)
train_vec = encode_w2v(train["text_tokens"], w2v_encoder)

tokenizer, model, device = load_model()
train_trans = encode_trans(train["text_clean_joined"], tokenizer, model, device)

train_label = train["label"]

In [ ]:
validation_bow = encode_bow(validation["text_tokens"], 
                            encoder=bow_encoder, 
                            return_bow=False)
validation_vec = encode_w2v(validation["text_tokens"], w2v_encoder)
validation_trans = encode_trans(validation["text_clean_joined"], tokenizer, model, device)

validation_label = validation["label"]

### <font color="#0ee071"> Encoded Text Vizualization <a class="anchor" id="P1.2.4"></a></font>

[Back to Index](#index)

As can be seen from visualisation all three encodings still are most definetly not linearaly separatable, therefore we should expect some difficulty for our models to learn.

In [ ]:
umap_object = umap.UMAP(n_neighbors=50, min_dist=0.9, random_state=SEED) 
umap_embedding = umap_object.fit_transform(train_bow)
visualize_dimensionality_reduction(umap_embedding, 
                                   train_label, 
                                   save_to=r"plots/bow_umap.png",
                                   title="Bag of Words Train Dataset - Dimentionality Reduction")

In [ ]:
umap_object = umap.UMAP(n_neighbors=50, min_dist=0.1, random_state=SEED) 
umap_embedding = umap_object.fit_transform(train_vec)
visualize_dimensionality_reduction(umap_embedding, 
                                   train_label,
                                   save_to=r"plots/w2v_umap.png",
                                   title="Word2Vec Train Dataset - Dimentionality Reduction")

In [ ]:
umap_object = umap.UMAP(n_neighbors=50, min_dist=0.1, random_state=SEED) 
umap_embedding = umap_object.fit_transform(train_trans)
visualize_dimensionality_reduction(umap_embedding, 
                                   train_label,
                                   save_to=r"plots/trans_umap.png",
                                   title="Transformer Train Dataset - Dimentionality Reduction")

# <font color="#0ee071"> II. Classification Models <a class="anchor" id="P2"></a></font>

[Back to Index](#index)

In this chapter we decided to try two models for each encoding we used. The two algorithms of choice were **K-Nearest Neigbours** and **Random Forest**. For each model we run a Hyperparameter Tuning with Random Search with 30 iterations.

## <font color="#0ee071"> Bag of Words Data <a class="anchor" id="P2.1"></a></font>

[Back to Index](#index)


### <font color="#3676FF"> KNN Model <a class="anchor" id="P2.1.1"></a></font>

[Back to Index](#index)

In [ ]:
best_score = -float("inf")

for g in tqdm.tqdm(ParameterSampler(KNN_GRID, n_iter=30, random_state=SEED)):
    _knn = KNeighborsClassifier()
    _knn.set_params(**g)
    _knn.fit(train_bow, train_label)

    new_score = recall_score(validation_label, 
                             _knn.predict(validation_bow), 
                             average="macro")
    # save if best
    if new_score > best_score:
        best_score = new_score
        best_grid = g

print("Best Recall: %0.5f" % best_score )
print("Grid:", best_grid)

In [ ]:
knn_classifier = KNeighborsClassifier()\
                .set_params(**best_grid)\
                .fit(train_bow, train_label)
knn_predicted = knn_classifier.predict(validation_bow)
knn_cm = confusion_matrix(validation_label, knn_predicted, normalize="true")

print(classification_report(validation_label, 
                            knn_predicted, 
                            target_names=("Bearish", "Bullish", "Neutral")))

### <font color="#3676FF"> Random Forest Model <a class="anchor" id="P2.1.2"></a></font>

[Back to Index](#index)

In [ ]:
best_score = -float("inf")

for g in tqdm.tqdm(ParameterSampler(RF_GRID, n_iter=30, random_state=SEED)):
    _rf = RandomForestClassifier()
    _rf.set_params(**g)
    _rf.fit(train_bow, train_label)

    new_score = recall_score(validation_label, 
                             _rf.predict(validation_bow), 
                             average="macro")
    # save if best
    if new_score > best_score:
        best_score = new_score
        best_grid = g

print("Best Recall: %0.5f" % best_score )
print("Grid:", best_grid)

In [ ]:
rf_classifier = RandomForestClassifier()\
                .set_params(**best_grid)\
                .fit(train_bow, train_label)

rf_predicted = rf_classifier.predict(validation_bow)
rf_cm = confusion_matrix(validation_label, rf_predicted, normalize="true")

print(classification_report(validation_label, 
                            rf_predicted, 
                            target_names=("Bearish", "Bullish", "Neutral")))

### <font color="#3676FF"> Model Analysis <a class="anchor" id="P2.1.3"></a></font>

[Back to Index](#index)

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(12, 5))

sns.heatmap(knn_cm,
            cmap="Reds",
            cbar=False,
            annot=True,
            fmt=".2%",
            xticklabels=("Bearish", "Bullish", "Neutral"), 
            yticklabels=("Bearish", "Bullish", "Neutral"),
            ax=ax[0]
            )
ax[0].set_title("KNN BOW Confusion Matrix")
ax[0].set_xlabel("Predicted Label")
ax[0].set_ylabel("True Label")

sns.heatmap(rf_cm,
            cmap="Reds",
            cbar=False,
            annot=True,
            fmt=".2%",
            xticklabels=("Bearish", "Bullish", "Neutral"), 
            yticklabels=("Bearish", "Bullish", "Neutral"),
            ax=ax[1]
            )
ax[1].set_title("RF BOW Confusion Matrix")
ax[1].set_xlabel("Predicted Label")
ax[1].set_ylabel("True Label")

plt.savefig("plots/bow_compare.png")
plt.show()

## <font color="#0ee071"> Word2Vec Data <a class="anchor" id="P2.2"></a></font>

[Back to Index](#index)

### <font color="#3676FF"> KNN Model <a class="anchor" id="P2.2.1"></a></font>

[Back to Index](#index)

In [ ]:
best_score = -float("inf")

for g in tqdm.tqdm(ParameterSampler(KNN_GRID, n_iter=30, random_state=SEED)):
    _knn = KNeighborsClassifier()
    _knn.set_params(**g)
    _knn.fit(train_vec, train_label)

    new_score = recall_score(validation_label, 
                             _knn.predict(validation_vec), 
                             average="macro")
    # save if best
    if new_score > best_score:
        best_score = new_score
        best_grid = g

print("Best Recall: %0.5f" % best_score )
print("Grid:", best_grid)

In [ ]:
knn_classifier = KNeighborsClassifier()\
                .set_params(**best_grid)\
                .fit(train_vec, train_label)
knn_predicted = knn_classifier.predict(validation_vec)
knn_cm = confusion_matrix(validation_label, knn_predicted, normalize="true")

print(classification_report(validation_label, 
                            knn_predicted, 
                            target_names=("Bearish", "Bullish", "Neutral")))

### <font color="#3676FF"> Random Forest Model <a class="anchor" id="P2.2.2"></a></font>

[Back to Index](#index)

In [ ]:
best_score = -float("inf")

for g in tqdm.tqdm(ParameterSampler(RF_GRID, n_iter=30, random_state=SEED)):
    _rf = RandomForestClassifier()
    _rf.set_params(**g)
    _rf.fit(train_vec, train_label)

    new_score = recall_score(validation_label, 
                             _rf.predict(validation_vec), 
                             average="macro")
    # save if best
    if new_score > best_score:
        best_score = new_score
        best_grid = g

print("Best Recall: %0.5f" % best_score )
print("Grid:", best_grid)

In [ ]:
rf_classifier = RandomForestClassifier()\
                .set_params(**best_grid)\
                .fit(train_vec, train_label)

rf_predicted = rf_classifier.predict(validation_vec)
rf_cm = confusion_matrix(validation_label, rf_predicted, normalize="true")

print(classification_report(validation_label, 
                            rf_predicted, 
                            target_names=("Bearish", "Bullish", "Neutral")))

### <font color="#3676FF"> Model Analysis <a class="anchor" id="P2.2.3"></a></font>

[Back to Index](#index)

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(12, 5))

sns.heatmap(knn_cm,
            cmap="Reds",
            cbar=False,
            annot=True,
            fmt=".2%",
            xticklabels=("Bearish", "Bullish", "Neutral"), 
            yticklabels=("Bearish", "Bullish", "Neutral"),
            ax=ax[0]
            )
ax[0].set_title("KNN W2V Confusion Matrix")
ax[0].set_xlabel("Predicted Label")
ax[0].set_ylabel("True Label")

sns.heatmap(rf_cm,
            cmap="Reds",
            cbar=False,
            annot=True,
            fmt=".2%",
            xticklabels=("Bearish", "Bullish", "Neutral"), 
            yticklabels=("Bearish", "Bullish", "Neutral"),
            ax=ax[1]
            )
ax[1].set_title("RF W2V Confusion Matrix")
ax[1].set_xlabel("Predicted Label")
ax[1].set_ylabel("True Label")

plt.savefig("plots/w2v_compare.png")
plt.show()

## <font color="#0ee071"> Transformers Data <a class="anchor" id="P2.3"></a></font>

[Back to Index](#index)

### <font color="#3676FF"> KNN Model <a class="anchor" id="P2.3.1"></a></font>

[Back to Index](#index)

In [ ]:
best_score = -float("inf")

for g in tqdm.tqdm(ParameterSampler(KNN_GRID, n_iter=30, random_state=SEED)):
    _knn = KNeighborsClassifier()
    _knn.set_params(**g)
    _knn.fit(train_trans, train_label)

    new_score = recall_score(validation_label, 
                             _knn.predict(validation_trans), 
                             average="macro")
    # save if best
    if new_score > best_score:
        best_score = new_score
        best_grid = g

print("Best Recall: %0.5f" % best_score )
print("Grid:", best_grid)

In [ ]:
knn_classifier = KNeighborsClassifier()\
                .set_params(**best_grid)\
                .fit(train_trans, train_label)
knn_predicted = knn_classifier.predict(validation_trans)
knn_cm = confusion_matrix(validation_label, knn_predicted, normalize="true")

print(classification_report(validation_label, 
                            knn_predicted, 
                            target_names=("Bearish", "Bullish", "Neutral")))

### <font color="#3676FF"> Random Forest Model <a class="anchor" id="P2.3.2"></a></font>

[Back to Index](#index)

In [ ]:
best_score = -float("inf")

for g in tqdm.tqdm(ParameterSampler(RF_GRID, n_iter=30, random_state=SEED)):
    _rf = RandomForestClassifier()
    _rf.set_params(**g)
    _rf.fit(train_trans, train_label)

    new_score = recall_score(validation_label, 
                             _rf.predict(validation_trans), 
                             average="macro")
    # save if best
    if new_score > best_score:
        best_score = new_score
        best_grid = g

print("Best Recall: %0.5f" % best_score )
print("Grid:", best_grid)

In [ ]:
rf_classifier = RandomForestClassifier()\
                .set_params(**best_grid)\
                .fit(train_trans, train_label)

rf_predicted = rf_classifier.predict(validation_trans)
rf_cm = confusion_matrix(validation_label, rf_predicted, normalize="true")

print(classification_report(validation_label, 
                            rf_predicted, 
                            target_names=("Bearish", "Bullish", "Neutral")))

### <font color="#3676FF"> Model Analysis <a class="anchor" id="P2.3.3"></a></font>

[Back to Index](#index)

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(12, 5))

sns.heatmap(knn_cm,
            cmap="Reds",
            cbar=False,
            annot=True,
            fmt=".2%",
            xticklabels=("Bearish", "Bullish", "Neutral"), 
            yticklabels=("Bearish", "Bullish", "Neutral"),
            ax=ax[0]
            )
ax[0].set_title("KNN Transformers Confusion Matrix")
ax[0].set_xlabel("Predicted Label")
ax[0].set_ylabel("True Label")

sns.heatmap(rf_cm,
            cmap="Reds",
            cbar=False,
            annot=True,
            fmt=".2%",
            xticklabels=("Bearish", "Bullish", "Neutral"), 
            yticklabels=("Bearish", "Bullish", "Neutral"),
            ax=ax[1]
            )
ax[1].set_title("RF Transformers Confusion Matrix")
ax[1].set_xlabel("Predicted Label")
ax[1].set_ylabel("True Label")

plt.savefig("plots/trans_compare.png")
plt.show()

## <font color="#0ee071"> Transformer Classifier <a class="anchor" id="P2.4"></a></font>

[Back to Index](#index)

In [ ]:
transformer_classifier = train_model(train,
                                     validation)

In [ ]:
trans_predicted = predict(validation.text.to_list(), 
                          transformer_classifier)
trans_cm = confusion_matrix(validation_label, trans_predicted, normalize="true")

print(classification_report(validation_label, 
                            trans_predicted, 
                            target_names=("Bearish", "Bullish", "Neutral")))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sns.heatmap(trans_cm,
            cmap="Reds",
            cbar=False,
            annot=True,
            fmt=".2%",
            xticklabels=("Bearish", "Bullish", "Neutral"), 
            yticklabels=("Bearish", "Bullish", "Neutral"),
            ax=ax
            )
ax.set_title("Transformer Classifier Confusion Matrix")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")

plt.savefig("plots/trans_classifier.png")
plt.show()